In [6]:
!pip install -q category_encoders xgboost

In [7]:
import glob, os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import category_encoders as ce
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

SEED, REPEATS, QUICK = 42, 5, False          # QUICK=True -> 12k-row smoke test only
OUT = "/kaggle/working/results_baselines"; os.makedirs(OUT, exist_ok=True)
CAT_COLS = ["dom", "tld"]

hits = glob.glob("/kaggle/input/**/*.csv", recursive=True)
DATA_FILE = next((f for f in hits if os.path.basename(f).lower() == "dataset.csv"), hits[0])
print("Using:", DATA_FILE)

Using: /kaggle/input/datasets/ummehabibamoushi/new-dataset-phising/Dataset.csv


In [8]:
# ---- load + clean (same order as the paper) ----
df = pd.read_csv(DATA_FILE)
df["tld"] = df["tld"].fillna(df["tld"].mode()[0])
df = df.drop_duplicates().reset_index(drop=True)
if QUICK: df = df.sample(min(12000, len(df)), random_state=SEED).reset_index(drop=True); REPEATS = 2
y = df["label"].astype(int)
NUM = [c for c in df.columns if c not in ("url", "dom", "tld", "label")]
X = df[NUM + CAT_COLS + ["url"]].copy(); X["grp"] = df["dom"]     # grp = domain, used only for splitting / inner folds
print(len(df), "URLs |", int(y.sum()), "phishing |", len(NUM), "numeric features")

115231 URLs | 16590 phishing | 22 numeric features


In [9]:
# ---- helpers (identical to the main notebook) ----
class NestedTargetEncoder(BaseEstimator, TransformerMixin):
    """Out-of-fold smoothed target encoding; inner folds grouped by domain (grouped=True)."""
    def __init__(self, cat_cols, group_col="grp", n_splits=5, smoothing=10, grouped=True, seed=SEED):
        self.cat_cols, self.group_col = cat_cols, group_col
        self.n_splits, self.smoothing, self.grouped, self.seed = n_splits, smoothing, grouped, seed
    def _enc(self): return ce.TargetEncoder(cols=self.cat_cols, smoothing=self.smoothing)
    def fit(self, Xd, yd):
        self.enc_ = self._enc().fit(Xd[self.cat_cols], yd); return self
    def fit_transform(self, Xd, yd):
        Xd = Xd.reset_index(drop=True); yd = pd.Series(np.asarray(yd)).reset_index(drop=True)
        out = np.zeros((len(Xd), len(self.cat_cols)))
        for a, b in GroupKFold(self.n_splits).split(Xd, yd, groups=Xd[self.group_col]):
            out[b] = self._enc().fit(Xd.loc[a, self.cat_cols], yd.loc[a]).transform(Xd.loc[b, self.cat_cols]).to_numpy()
        self.fit(Xd, yd); return out
    def transform(self, Xd): return self.enc_.transform(Xd[self.cat_cols]).to_numpy()

def metrics(yt, pred, sc):
    tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    return dict(acc=(tp+tn)/len(yt), phish_prec=precision_score(yt, pred, zero_division=0), phish_rec=recall_score(yt, pred),
                phish_f1=f1_score(yt, pred), bal_acc=balanced_accuracy_score(yt, pred), mcc=matthews_corrcoef(yt, pred),
                roc_auc=roc_auc_score(yt, sc), pr_auc=average_precision_score(yt, sc), TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn))

def group_split(seed, test_size=0.2):
    return next(GroupShuffleSplit(1, test_size=test_size, random_state=seed).split(X, y, groups=X["grp"]))

def tab_pipe(model):
    prep = ColumnTransformer([("te", NestedTargetEncoder(CAT_COLS), CAT_COLS + ["grp"]), ("sc", StandardScaler(), NUM)])
    return Pipeline([("prep", prep), ("model", model)])

def text_pipe(clf):
    return Pipeline([("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), sublinear_tf=True, min_df=2,
                                               max_features=300000, lowercase=True)), ("model", clf)])

def run_baseline(name, tr, te, seed):
    ytr = y.iloc[tr]; text = name.startswith("CharNgram")
    if name == "HGB":      pipe = tab_pipe(HistGradientBoostingClassifier(class_weight="balanced", random_state=seed))
    elif name == "XGBoost": pipe = tab_pipe(XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                    scale_pos_weight=float((ytr == 0).sum()/max((ytr == 1).sum(), 1)),
                                    eval_metric="logloss", random_state=seed, n_jobs=-1))
    elif name == "CharNgram-LR":  pipe = text_pipe(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
    elif name == "CharNgram-SVM": pipe = text_pipe(LinearSVC(class_weight="balanced", random_state=seed))
    Xtr, Xte = (X.iloc[tr]["url"], X.iloc[te]["url"]) if text else (X.iloc[tr], X.iloc[te])
    pipe.fit(Xtr, ytr); pred = pipe.predict(Xte)
    m = pipe.named_steps["model"]
    Z = pipe.named_steps["tfidf"].transform(Xte) if text else pipe.named_steps["prep"].transform(Xte)
    sc = m.predict_proba(Z)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Z)
    return metrics(y.iloc[te], pred, sc)

NAMES = ["HGB"] + (["XGBoost"] if XGBClassifier else []) + ["CharNgram-LR", "CharNgram-SVM"]
print("baselines:", NAMES)

baselines: ['HGB', 'XGBoost', 'CharNgram-LR', 'CharNgram-SVM']


## Headline split (seed 42, same split as the paper)

In [10]:
tr, te = group_split(SEED)
rows = []
for n in NAMES:
    mt = run_baseline(n, tr, te, SEED); rows.append({"model": n, **mt}); print(n, round(mt["acc"], 4), round(mt["phish_rec"], 4), round(mt["phish_f1"], 4))
head = pd.DataFrame(rows); head.to_csv(f"{OUT}/baselines_headline_split.csv", index=False)
head[["model", "acc", "phish_prec", "phish_rec", "phish_f1", "bal_acc", "mcc", "pr_auc", "TP", "TN", "FP", "FN"]].round(4)

HGB 0.9656 0.9204 0.9278
XGBoost 0.9684 0.9278 0.9338
CharNgram-LR 0.9809 0.9875 0.9613
CharNgram-SVM 0.9866 0.9532 0.9717


,model,acc,phish_prec,phish_rec,phish_f1,bal_acc,mcc,pr_auc,TP,TN,FP,FN
0,HGB,0.9656,0.9352,0.9204,0.9278,0.9501,0.9052,0.9807,5747,19338,398,497
1,XGBoost,0.9684,0.9398,0.9278,0.9338,0.9545,0.9130,0.9809,5793,19365,371,451
2,CharNgram-LR,0.9809,0.9365,0.9875,0.9613,0.9832,0.9493,0.9931,6166,19318,418,78
3,CharNgram-SVM,0.9866,0.9908,0.9532,0.9717,0.9752,0.9633,0.9970,5952,19681,55,292


## Repeated domain-grouped splits (seeds 0 to REPEATS-1)

In [11]:
rep = []
for s in range(REPEATS):
    tr_, te_ = group_split(s)
    for n in NAMES:
        mt = run_baseline(n, tr_, te_, s); rep.append({"seed": s, "model": n, **mt}); print("seed", s, n, round(mt["phish_f1"], 4))
R = pd.DataFrame(rep); R.to_csv(f"{OUT}/baselines_repeated_raw.csv", index=False)
cols = ["acc", "phish_prec", "phish_rec", "phish_f1", "bal_acc", "mcc", "pr_auc"]
summ = R.groupby("model")[cols].agg(["mean", "std"]); summ.to_csv(f"{OUT}/baselines_repeated_summary.csv")
summ.round(4)

seed 0 HGB 0.9192
seed 0 XGBoost 0.9266
seed 0 CharNgram-LR 0.9549
seed 0 CharNgram-SVM 0.9323
seed 1 HGB 0.927
seed 1 XGBoost 0.9326
seed 1 CharNgram-LR 0.9549
seed 1 CharNgram-SVM 0.9332
seed 2 HGB 0.9449
seed 2 XGBoost 0.9276
seed 2 CharNgram-LR 0.964
seed 2 CharNgram-SVM 0.9715
seed 3 HGB 0.911
seed 3 XGBoost 0.9098
seed 3 CharNgram-LR 0.9667
seed 3 CharNgram-SVM 0.9777
seed 4 HGB 0.8578
seed 4 XGBoost 0.8567
seed 4 CharNgram-LR 0.8884
seed 4 CharNgram-SVM 0.9233


acc         phish_prec         phish_rec         phish_f1         bal_acc             mcc          pr_auc        
                 mean     std       mean     std      mean     std     mean     std    mean     std    mean     std    mean     std
model                                                                                                                              
CharNgram-LR   0.9847  0.0030     0.9381  0.0281    0.9537  0.0378   0.9458  0.0325  0.9713  0.0183  0.9366  0.0328  0.9831  0.0181
CharNgram-SVM  0.9848  0.0079     0.9889  0.0023    0.9105  0.0454   0.9476  0.0250  0.9543  0.0225  0.9399  0.0268  0.9904  0.0089
HGB            0.9727  0.0077     0.8898  0.0489    0.9366  0.0310   0.9120  0.0328  0.9582  0.0159  0.8962  0.0324  0.9743  0.0170
XGBoost        0.9724  0.0065     0.8930  0.0534    0.9312  0.0354   0.9107  0.0314  0.9559  0.0162  0.8949  0.0302  0.9736  0.0179

In [12]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
print(head[["model", "acc", "phish_prec", "phish_rec", "phish_f1", "bal_acc", "mcc", "TP", "TN", "FP", "FN"]].round(4).to_string())
print(summ.round(4).to_string())

           model     acc  phish_prec  phish_rec  phish_f1  bal_acc     mcc    TP     TN   FP   FN
0            HGB  0.9656      0.9352     0.9204    0.9278   0.9501  0.9052  5747  19338  398  497
1        XGBoost  0.9684      0.9398     0.9278    0.9338   0.9545  0.9130  5793  19365  371  451
2   CharNgram-LR  0.9809      0.9365     0.9875    0.9613   0.9832  0.9493  6166  19318  418   78
3  CharNgram-SVM  0.9866      0.9908     0.9532    0.9717   0.9752  0.9633  5952  19681   55  292
                  acc         phish_prec         phish_rec         phish_f1         bal_acc             mcc          pr_auc        
                 mean     std       mean     std      mean     std     mean     std    mean     std    mean     std    mean     std
model                                                                                                                              
CharNgram-LR   0.9847  0.0030     0.9381  0.0281    0.9537  0.0378   0.9458  0.0325  0.9713  0.0183  0.9366  0.032